In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np


In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import os
import logging
import time

# Create Logging Module

logging.basicConfig(
    filename ="logs/ingestion_db.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"

)

# Create Connection With MySQL

user = "root"
password = "sanket1234"
host = "localhost"
port = "3306"
database =  "delivery_performance"


try:
    engine = create_engine(
        f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    )
except:
    print("Connection Error")

In [ ]:
df = pd.read_sql_query("""select * from product_delivery_summary""", con = engine)

Create A CSV of Clean Data And Save It As df And Then Run The Code

In [ ]:
x = df[df['order_delivered_customer_days']> 0 ] ['order_delivered_customer_days']
bin_egde = np.arange(0, 45, 5)
plt.figure(figsize = (10, 6))
ax = sns.histplot(x, bins = bin_egde, color = 'skyblue', edgecolor = 'black')
ax.bar_label(ax.containers[0])
plt.xticks(bin_egde)
plt.title('Distribution of Order-to-Delivery Lead Time')
plt.xlabel('Delivery Duration (Days)')
plt.ylabel('Total Orders')
plt.grid(axis = 'y', alpha = 0.3)
plt.show()

In [ ]:
df['delay'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

In [ ]:
df[df['delay']> 0 ]['delay']

In [ ]:
(df['delay'] > 0).sum()

In [ ]:
x = df[df['delay']> 0 ] ['delay']
bin_egde = np.arange(0, 45, 5)
plt.figure(figsize = (10, 6))
ax = sns.histplot(x, bins = bin_egde, color = 'skyblue', edgecolor = 'black')
ax.bar_label(ax.containers[0])
plt.xticks(bin_egde)
plt.title('Distribution of Order TO Delivery Delay To Customer ')
plt.xlabel('Delivery Delay(Duration) Days')
plt.ylabel('Total Orders Delay')
plt.grid(axis='y', alpha=0.2)
plt.show()


In [ ]:
status_count = df['order_delivered_late'].value_counts()

labels = ['Late' if i == 1 else 'On Time' for i in status_count.index]

plt.figure(figsize = (8, 6))
plt.pie(status_count , labels = labels, autopct = '%1.1f%%', startangle = 140,colors = [ '#66b3ff', '#ff9999'], explode = (0.05, 0), shadow = True)
plt.title('Percentage (Order Delivery) : On Time vs Late')
plt.axis('equal')
plt.show()

In [ ]:
late_shipping = df['is_late_shipping'].value_counts()
labels = ['Late' if i == 1 else 'On Time' for i in late_shipping.index]
plt.figure(figsize = (8, 6))
plt.pie(late_shipping, labels = labels, autopct = '%1.1f%%', startangle = 140, colors = ['#66b3ff','#ff9999'], shadow = True)
plt.axis('equal')
plt.title('Percentage (Order Shipping) : On Time vs Late Shipping')
plt.show()

In [ ]:
x = df[df['order_handling_days']> 0 ] ['order_handling_days']
bin_egde = np.arange(0, 45, 5)
plt.figure(figsize = (10, 6))
ax = sns.histplot(x, bins = bin_egde, color = 'skyblue', edgecolor = 'black')
ax.bar_label(ax.containers[0])
plt.xticks(bin_egde)
plt.title('Distribution : Order Handling By Sellers')
plt.xlabel('Duration of Order Handling (Days)')
plt.ylabel('Total Order Handling')
plt.grid(axis='y', alpha=0.2)
plt.show()

In [ ]:
late_shipping = df[df['is_late_shipping'] == 1]
top_10_late_sellers = late_shipping['seller_id'].value_counts().head(10).reset_index()
top_10_late_sellers.columns = ['seller_id', 'late_count']
plt.figure(figsize = (12, 6))
sns.barplot(data = top_10_late_sellers, x = 'seller_id', y = "late_count", palette = 'Reds_r')
plt.title('Top 10 Sellers With Most Late Shipments')
plt.xlabel('Seller ID')
plt.ylabel('Number of Late Shipments')
plt.xticks(rotation = 90)
plt.show()

In [ ]:
sns.set_theme(style = 'whitegrid')

plt.figure(figsize = (10, 6))
sns.scatterplot(data = df, x = 'total_freight_value', y = 'order_delivered_customer_days', alpha = 0.6, color = 'teal')
plt.title('Relationship Between Total Freight Price and Delivery Time')
plt.xlabel('Freight Price')
plt.ylabel('Delivery To Customer Days')
plt.show()

In [ ]:
sns.set_theme(style = 'whitegrid')

plt.figure(figsize = (10, 6))
sns.barplot(data = df, x = 'payment_type', y = 'order_approved_days', palette = 'viridis', errorbar = None )

plt.title('Average Order Approval Days by Payment Type')
plt.xlabel('Payment Type')
plt.ylabel('Average Days to Approve')

plt.show()

In [ ]:
top_5_late = pd.read_sql_query('''select 
	product_category, 
    count(*) as late_order_delivered
from product_delivery_summary 
where order_delivered_late is true 
group by product_category 
order by late_order_delivered desc limit 5;''', con = engine)

ax = sns.barplot(data = top_5_late , x = 'product_category', y = 'late_order_delivered', palette = 'viridis')
for container in ax.containers:
    ax.bar_label(container, padding = 3)
plt.title('Top 5 Product Categories with Late Deliveries')
plt.xlabel('Product Category')
plt.ylabel('Number of Late Shipments')
plt.xticks(rotation = 60)
plt.show()


In [ ]:
corr_matrix = df.select_dtypes(include=['number']).corr()

plt.figure(figsize=(12, 8))

sns.heatmap(
    corr_matrix, 
    annot=True,          
    fmt=".2f",           
    cmap='coolwarm',     
    center=0,           
    square=True,         
    linewidths=0.5       
)

plt.title('Numerical Correlation Heatmap')
plt.show()
